In [1]:

# Step 1: Install the data tools
!pip -q install datasets pandas

# Step 2: Import packages
import pandas as pd
from datasets import load_dataset

# Step 3: Connect to MerRec
dataset = load_dataset(
    "mercari-us/merrec",
    split="train",
    streaming=True
)

# Step 4: Read only the first 10,000 records
sample = list(dataset.take(10000))

# Step 5: Turn them into a table
df = pd.DataFrame(sample)

# Step 6: Inspect the data
print("Dataset shape:", df.shape)

print("\nColumn names:")
print(df.columns.tolist())

print("\nFirst five rows:")
display(df.head())

print("\nEvent types:")
print(df["event_id"].value_counts())

print("\nProduct categories:")
print(df["c1_name"].value_counts().head(10))

print("\nMissing values:")
print(df.isnull().sum())


README.md:   0%|          | 0.00/4.04k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/2170 [00:00<?, ?it/s]

Dataset shape: (10000, 25)

Column names:
['user_id', 'stime', 'session_id', 'sequence_id', 'sequence_length', 'event_id', 'item_id', 'product_id', 'name', 'price', 'c0_name', 'c0_id', 'c1_name', 'c1_id', 'c2_name', 'c2_id', 'brand_name', 'brand_id', 'item_condition_id', 'item_condition_name', 'size_name', 'size_id', 'color', 'shipper_id', 'shipper_name']

First five rows:


,user_id,stime,session_id,sequence_id,sequence_length,event_id,item_id,product_id,name,price,...,c2_id,brand_name,brand_id,item_condition_id,item_condition_name,size_name,size_id,color,shipper_id,shipper_name
0,45,2023-05-04 19:21:12,e81215462fcaff69cfa87276e2f47e5815b638ba12ba9b...,20230501_0,1,item_view,238357102,10548_1332,Kaleidos Comet Catcher Highlighter,55.0,...,1332.0,Indie,10548,1,New,None,NaN,None,1,Buyer
1,50,2023-05-02 21:58:49,b54f49c79fe63a91cf3c6bcbdeb4f4e54c0fb3f98da300...,20230501_0,1,item_view,153080686,9820_923,Gourmia French Door Xl Air Fryer Oven,117.0,...,923.0,Hamilton Beach,9820,1,New,None,NaN,None,1,Buyer
2,56,2023-05-11 23:57:15,7100e5ec0c89a773dcd53d8b36f8b35b1507d2411ba240...,20230501_0,2,item_view,228645576,0_1160,Gold Heart Pendant Necklace,14.0,...,1160.0,None,0,1,New,None,NaN,None,1,Buyer
3,56,2023-05-11 23:57:17,7100e5ec0c89a773dcd53d8b36f8b35b1507d2411ba240...,20230501_0,2,item_view,90449801,12630_1149,Lululemon Hotty Hot Shorts Low Rise,50.0,...,1149.0,lululemon athletica,12630,2,Like new,M (8-10),1.0,None,1,Buyer
4,79,2023-05-19 23:56:20,37b20d4b650ba7cce590f85c1bf1f93122b06401016bc7...,20230501_0,1,item_view,192635445,15928_1314,Nintendo 3DS XL PIKACHU YELLOW EDITION,2000.0,...,1314.0,Nintendo,15928,1,New,None,NaN,yellow,2,Seller



Event types:
event_id
item_view               8964
item_like                849
item_add_to_cart_tap     131
offer_make                39
buy_start                 12
buy_comp                   5
Name: count, dtype: int64

Product categories:
c1_name
Jewelry                    2019
Women's handbags           1112
Shoes                       600
Makeup                      521
Women's accessories         492
Collectibles & Hobbies      397
Tops & blouses              363
Stuffed Animals & Plush     363
Dolls & Accessories         309
Skin care                   233
Name: count, dtype: int64

Missing values:
user_id                   0
stime                     0
session_id                0
sequence_id               0
sequence_length           0
event_id                  0
item_id                   0
product_id                0
name                      0
price                     0
c0_name                   0
c0_id                     0
c1_name                   0
c1_id                

In [2]:

# EXPERIMENT 2: Understanding our products

# 1. How many unique products and users?
print("Unique products:", df["product_id"].nunique())
print("Unique items:", df["item_id"].nunique())
print("Unique users:", df["user_id"].nunique())
print("Unique categories:", df["c1_name"].nunique())


# 2. How many events does each product have?
product_summary = (
    df.groupby(["c1_name", "product_id"])
      .agg(
          event_count=("event_id", "size"),
          view_count=("event_id", lambda x: (x == "item_view").sum()),
          like_count=("event_id", lambda x: (x == "item_like").sum()),
          item_count=("item_id", "nunique"),
          avg_price=("price", "mean")
      )
      .reset_index()
)

print("\nProduct-level dataset:")
display(product_summary.head(10))

print("\nNumber of product-category combinations:",
      len(product_summary))


# 3. Does one product appear in multiple categories?
category_per_product = (
    df.groupby("product_id")["c1_name"].nunique()
)

print("\nProducts appearing in multiple categories:",
      (category_per_product > 1).sum())


# 4. How many item IDs belong to each product?
print("\nItem IDs per product:")
print(df.groupby("product_id")["item_id"].nunique().describe())


# 5. Check the time range
print("\nTime column information:")
print(df["stime"].dtype)
print("Earliest recorded value:", df["stime"].min())
print("Latest recorded value:", df["stime"].max())


# 6. Check the distribution of products by category
print("\nProducts per category:")
display(
    product_summary.groupby("c1_name")["product_id"]
    .nunique()
    .sort_values(ascending=False)
    .head(15)
)


Unique products: 2942
Unique items: 7739
Unique users: 93
Unique categories: 155

Product-level dataset:


,c1_name,product_id,event_count,view_count,like_count,item_count,avg_price
0,Accessories,0_1376,2,1,1,1,8.000000
1,Accessories,0_1379,10,7,3,7,17.850000
2,Accessories,0_1381,2,2,0,2,5.000000
3,Accessories,0_1382,1,1,0,1,30.000000
4,Accessories,0_1385,13,10,3,8,11.346154
5,Accessories,0_1388,2,1,1,1,22.000000
6,Accessories,0_1655,49,20,28,37,13.365306
7,Accessories,0_1659,1,1,0,1,10.000000
8,Accessories,0_416,3,3,0,3,7.203333
9,Accessories,10235_1655,3,3,0,3,17.333333



Number of product-category combinations: 2942

Products appearing in multiple categories: 0

Item IDs per product:
count    2942.000000
mean        2.630523
std         6.470801
min         1.000000
25%         1.000000
50%         1.000000
75%         2.000000
max        98.000000
Name: item_id, dtype: float64

Time column information:
datetime64[ns]
Earliest recorded value: 2023-05-01 00:01:29
Latest recorded value: 2023-05-30 23:42:54

Products per category:


,product_id
c1_name,
Jewelry,349
Women's handbags,243
Shoes,190
Tops & blouses,169
Women's accessories,158
Makeup,99
Collectibles & Hobbies,89
Athletic apparel,85
Coats & jackets,84


In [3]:

# EXPERIMENT 3: Create a shuffled pilot sample

from datasets import load_dataset
import pandas as pd
import numpy as np

# 1. Reload the dataset
stream = load_dataset(
    "mercari-us/merrec",
    split="train",
    streaming=True
)

# 2. Shuffle records before sampling
shuffled_stream = stream.shuffle(
    seed=42,
    buffer_size=100000
)

# 3. Take 10,000 records
random_records = list(shuffled_stream.take(10000))
df_random = pd.DataFrame(random_records)

# 4. Compare with our original sample
print("Original sample:")
print("Users:", df["user_id"].nunique())
print("Products:", df["product_id"].nunique())
print("Categories:", df["c1_name"].nunique())

print("\nShuffled sample:")
print("Users:", df_random["user_id"].nunique())
print("Products:", df_random["product_id"].nunique())
print("Categories:", df_random["c1_name"].nunique())

# 5. Check the new time range
print("\nTime range:")
print(df_random["stime"].min())
print(df_random["stime"].max())

# 6. Check event distribution
print("\nEvent distribution:")
print(df_random["event_id"].value_counts(normalize=True))

# 7. Check category distribution
print("\nTop 10 categories:")
print(df_random["c1_name"].value_counts().head(10))


Resolving data files:   0%|          | 0/2170 [00:00<?, ?it/s]

Original sample:
Users: 93
Products: 2942
Categories: 155

Shuffled sample:
Users: 801
Products: 4882
Categories: 190

Time range:
2023-08-01 00:06:50
2023-08-30 23:53:27

Event distribution:
event_id
item_view               0.8418
item_like               0.1400
item_add_to_cart_tap    0.0115
offer_make              0.0057
buy_start               0.0009
buy_comp                0.0001
Name: proportion, dtype: float64

Top 10 categories:
c1_name
Shoes                           612
Tops & blouses                  559
Athletic apparel                487
Women's handbags                420
Makeup                          408
Collectibles & Hobbies          403
Dresses                         335
Action Figures & Accessories    333
Fragrance                       311
Jewelry                         277
Name: count, dtype: int64


In [4]:

# EXPERIMENT 4: Check product-level consistency

product_check = (
    df_random.groupby("product_id")
    .agg(
        item_count=("item_id", "nunique"),
        category_count=("c1_name", "nunique"),
        price_variation=("price", "nunique"),
        brand_variation=("brand_name", "nunique"),
        condition_variation=("item_condition_name", "nunique")
    )
    .reset_index()
)

print("Product consistency check:")
display(product_check.head(10))

print("\nProducts with multiple item IDs:",
      (product_check["item_count"] > 1).sum())

print("Products with multiple prices:",
      (product_check["price_variation"] > 1).sum())

print("Products with multiple brands:",
      (product_check["brand_variation"] > 1).sum())

print("Products with multiple conditions:",
      (product_check["condition_variation"] > 1).sum())


Product consistency check:


,product_id,item_count,category_count,price_variation,brand_variation,condition_variation
0,0_1001,9,1,8,0,2
1,0_1002,5,1,5,0,1
2,0_1004,1,1,1,0,1
3,0_1006,2,1,2,0,1
4,0_1011,9,1,7,0,2
5,0_1013,2,1,2,0,1
6,0_1018,1,1,1,0,1
7,0_1019,1,1,1,0,1
8,0_1025,2,1,2,0,2
9,0_1026,2,1,2,0,2



Products with multiple item IDs: 1271
Products with multiple prices: 1256
Products with multiple brands: 0
Products with multiple conditions: 916


In [5]:
# 只保留 item_view
views = df_random[df_random["event_id"] == "item_view"].copy()

# 计算每个 listing 获得的 views
item_views = (
    views.groupby(["product_id", "item_id"])
    .size()
    .reset_index(name="view_count")
)

# 看看每个 product 下有多少 listing
product_view_summary = (
    item_views.groupby("product_id")
    .agg(
        item_count=("item_id", "nunique"),
        total_views=("view_count", "sum"),
        max_item_views=("view_count", "max")
    )
    .reset_index()
)

# 一个 product 中，views 最多的 listing 占该 product 总 views 的比例
product_view_summary["top_item_share"] = (
    product_view_summary["max_item_views"] /
    product_view_summary["total_views"]
)

print(product_view_summary.describe())

        item_count  total_views  max_item_views  top_item_share
count  4472.000000  4472.000000     4472.000000     4472.000000
mean      1.814624     1.882379        1.062835        0.844225
std       2.671424     2.808832        0.306984        0.282944
min       1.000000     1.000000        1.000000        0.025000
25%       1.000000     1.000000        1.000000        1.000000
50%       1.000000     1.000000        1.000000        1.000000
75%       1.000000     2.000000        1.000000        1.000000
max      46.000000    58.000000        6.000000        1.000000


In [6]:
# Listing-level ranking
item_rank = (
    item_views.sort_values("view_count", ascending=False)
    .reset_index(drop=True)
)

# Product-level ranking
product_rank = (
    item_views.groupby("product_id")["view_count"]
    .sum()
    .sort_values(ascending=False)
    .reset_index()
)

print("Top 10 listings:")
print(item_rank.head(10))

print("\nTop 10 products:")
print(product_rank.head(10))

Top 10 listings:
   product_id    item_id  view_count
0   2388_2175   42871367           6
1  21237_2175  153337183           6
2      0_2175  221144805           5
3      0_2909  176541662           5
4     802_973  114969183           4
5  15468_1159   34328692           4
6  16600_1645    3656476           4
7    815_1170  139526532           4
8    9739_766  233478727           4
9  10235_2150  121416794           4

Top 10 products:
   product_id  view_count
0      0_2175          58
1  15987_1224          40
2      0_1259          34
3   10525_576          34
4   11089_467          33
5   15568_568          32
6    2595_568          31
7   7969_2428          30
8  15928_1315          28
9       0_561          27


In [7]:
price_check = (
    df_random.groupby("product_id")
    .agg(
        item_count=("item_id", "nunique"),
        mean_price=("price", "mean"),
        median_price=("price", "median"),
        min_price=("price", "min"),
        max_price=("price", "max")
    )
    .reset_index()
)

price_check["price_range"] = (
    price_check["max_price"] - price_check["min_price"]
)

print(price_check.describe())
print("\nExamples with largest price differences:")
print(
    price_check.sort_values("price_range", ascending=False).head(10)
)

        item_count   mean_price  median_price    min_price    max_price  \
count  4882.000000  4882.000000   4882.000000  4882.000000  4882.000000   
mean      1.930971    55.868510     54.091105    47.598263    70.663589   
std       3.134352   123.583254    122.371761   113.977516   164.539294   
min       1.000000     1.000000      1.000000     1.000000     1.000000   
25%       1.000000    14.000000     13.035000    11.000000    15.000000   
50%       1.000000    24.000000     23.000000    20.000000    26.000000   
75%       2.000000    48.652206     45.000000    39.000000    60.000000   
max      48.000000  2000.000000   2000.000000  2000.000000  2995.000000   

       price_range  
count  4882.000000  
mean     23.065326  
std     115.630858  
min       0.000000  
25%       0.000000  
50%       0.000000  
75%       2.000000  
max    2495.000000  

Examples with largest price differences:
      product_id  item_count   mean_price  median_price  min_price  max_price  \
3891    4769

In [8]:
#ex6 spearman
# 每个 product group 的总 views 和最大单个 listing views
ranking_check = (
    item_views.groupby("product_id")
    .agg(
        total_product_views=("view_count", "sum"),
        max_listing_views=("view_count", "max"),
        listing_count=("item_id", "nunique")
    )
    .reset_index()
)

# 分别计算两种排序
ranking_check["rank_total"] = (
    ranking_check["total_product_views"]
    .rank(method="average", ascending=False)
)

ranking_check["rank_max_listing"] = (
    ranking_check["max_listing_views"]
    .rank(method="average", ascending=False)
)

# Spearman rank correlation
correlation = ranking_check[
    ["rank_total", "rank_max_listing"]
].corr(method="spearman").iloc[0, 1]

print("Spearman rank correlation:", correlation)
print("\nLargest differences between the two ranks:")

ranking_check["rank_difference"] = (
    ranking_check["rank_total"] - ranking_check["rank_max_listing"]
).abs()

print(
    ranking_check.sort_values("rank_difference", ascending=False).head(10)
)

Spearman rank correlation: 0.40043148968577696

Largest differences between the two ranks:
      product_id  total_product_views  max_listing_views  listing_count  \
1586  15987_1224                   40                  1             40   
84        0_1259                   34                  1             34   
1492   15568_568                   32                  1             32   
3077    2595_568                   31                  1             31   
62        0_1176                   27                  1             27   
506        0_904                   26                  1             26   
1574  15987_1167                   26                  1             26   
4422    9814_561                   25                  1             25   
4247   8886_1176                   24                  1             24   
1100  12864_2430                   24                  1             24   

      rank_total  rank_max_listing  rank_difference  
1586         2.0            2

In [12]:
df_analysis = df_random.dropna(
    subset=["product_id", "c1_name"]
).copy()

print("Records used:", len(df_analysis))
print("Product groups:", df_analysis["product_id"].nunique())

Records used: 9988
Product groups: 4874


In [13]:
import pandas as pd

# One row per product group
base = (
    df_analysis.groupby(["product_id", "c1_name"])
    .agg(
        listing_count=("item_id", "nunique"),
        median_price=("price", "median"),
        price_range=("price", lambda x: x.max() - x.min()),
        brand=("brand_name", lambda x: x.dropna().mode().iloc[0]
               if not x.dropna().mode().empty else None),
        condition=("item_condition_name",
                   lambda x: x.dropna().mode().iloc[0]
                   if not x.dropna().mode().empty else None)
    )
    .reset_index()
)

# Aggregate views and likes
views = (
    df_analysis[df_analysis["event_id"] == "item_view"]
    .groupby(["product_id", "c1_name"])
    .size().reset_index(name="view_count")
)

likes = (
    df_analysis[df_analysis["event_id"] == "item_like"]
    .groupby(["product_id", "c1_name"])
    .size().reset_index(name="like_count")
)

# Combine
final_df = base.merge(
    views, on=["product_id", "c1_name"], how="left"
).merge(
    likes, on=["product_id", "c1_name"], how="left"
)

final_df[["view_count", "like_count"]] = (
    final_df[["view_count", "like_count"]].fillna(0)
)

# Category-normalized percentile rank
final_df["attention_percentile"] = (
    final_df.groupby("c1_name")["view_count"]
    .rank(pct=True)
)

# Remove source identifiers from the released aggregate table
final_df = final_df.drop(columns=["product_id"])
final_df = final_df.rename(columns={"c1_name": "category"})
final_df.insert(
    0, "product_group_id",
    ["PG" + str(i + 1) for i in range(len(final_df))]
)

final_df.to_csv("merrec_product_attention.csv", index=False)

print(final_df.shape)
print(final_df.head())

(4874, 10)
  product_group_id             category  listing_count  median_price  \
0              PG1               Makeup              9         15.65   
1              PG2               Makeup              5         21.00   
2              PG3            Skin care              1         15.00   
3              PG4  Tools & accessories              2         78.20   
4              PG5  Tools & accessories              9         13.00   

   price_range brand condition  view_count  like_count  attention_percentile  
0        106.0  None       New         7.0         2.0              0.974432  
1         41.0  None       New         5.0         0.0              0.928977  
2          0.0  None      Good         1.0         0.0              0.419753  
3        133.6  None       New         1.0         1.0              0.400000  
4         20.0  None       New         9.0         3.0              1.000000  


In [10]:
from google.colab import files

files.download("merrec_product_attention.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [11]:
print("Unique product IDs:", df_random["product_id"].nunique())
print("Missing product IDs:", df_random["product_id"].isna().sum())

print("Missing categories:", df_random["c1_name"].isna().sum())

print(
    "Unique product-category pairs:",
    df_random[["product_id", "c1_name"]].drop_duplicates().shape[0]
)

print("Final dataset rows:", len(final_df))

Unique product IDs: 4882
Missing product IDs: 0
Missing categories: 12
Unique product-category pairs: 4882
Final dataset rows: 4874
